# Zalo AI 2025 – Baseline Submission

This notebook builds a simple baseline to generate `submission.json` for the public test set.\n\n**Steps**:
1. Configure paths.
2. Load YOLO model from the `model/` folder.
3. Iterate over `data/public_test/samples/*/drone_video.mp4`.
4. Run detection and convert results to the required submission schema.
5. Save `submission.json` in the project root.

Dependencies (install if missing):
- `pip install ultralytics opencv-python`


In [ ]:
import os
import json
import glob

import cv2  # pip install opencv-python
from ultralytics import YOLO  # pip install ultralytics

# Paths (assuming this notebook is in the repo root)
PROJECT_ROOT = os.path.abspath('.')
DATA_DIR = os.path.join(PROJECT_ROOT, 'data')
PUBLIC_TEST_SAMPLES_DIR = os.path.join(DATA_DIR, 'public_test', 'samples')
MODEL_PATH = os.path.join(PROJECT_ROOT, 'model', 'yolov8n.pt')
OUTPUT_PATH = os.path.join(PROJECT_ROOT, 'submission.json')

print('PROJECT_ROOT:', PROJECT_ROOT)
print('PUBLIC_TEST_SAMPLES_DIR:', PUBLIC_TEST_SAMPLES_DIR)
print('MODEL_PATH:', MODEL_PATH)
print('OUTPUT_PATH:', OUTPUT_PATH)


In [ ]:
# Load YOLO model
assert os.path.isfile(MODEL_PATH), f'Model file not found: {MODEL_PATH}'
model = YOLO(MODEL_PATH)
# Optional: fuse model for slightly faster inference
try:
    model.fuse()
except Exception as e:
    print('Warning: model.fuse() failed:', e)

model


In [ ]:
def run_detection_on_video(video_path, model, conf=0.25):
    """
    Run YOLO detection on a single video and return a list in the format:
    [
        { 'bboxes': [ {frame, x1, y1, x2, y2}, ... ] }
    ]
    If no boxes are found, an empty list is returned (no detections).
    """
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise RuntimeError(f'Cannot open video: {video_path}')

    frame_idx = 0
    all_bboxes = []

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        # Run model on the current frame
        results = model(frame, conf=conf, verbose=False)[0]

        # Simple baseline: take at most one box per frame (the first one)
        if len(results.boxes) > 0:
            box = results.boxes[0]
            x1, y1, x2, y2 = box.xyxy[0].tolist()
            all_bboxes.append({
                'frame': int(frame_idx),
                'x1': int(x1),
                'y1': int(y1),
                'x2': int(x2),
                'y2': int(y2),
            })

        frame_idx += 1

    cap.release()

    if not all_bboxes:
        # No detections for this video
        return []

    # For simplicity, treat all frames as a single visible interval
    return [
        {
            'bboxes': all_bboxes
        }
    ]


# Quick smoke test on a single video (optional)
sample_videos = sorted(glob.glob(os.path.join(PUBLIC_TEST_SAMPLES_DIR, '*', 'drone_video.mp4')))
print('Found test videos:', len(sample_videos))
if sample_videos:
    print('Example video:', sample_videos[0])


In [ ]:
# Run inference on all public test videos and build submission.json
video_paths = sorted(glob.glob(os.path.join(PUBLIC_TEST_SAMPLES_DIR, '*', 'drone_video.mp4')))
assert video_paths, f'No videos found under {PUBLIC_TEST_SAMPLES_DIR}'

submission = []

for video_path in video_paths:
    # Folder name (e.g., 'BlackBox_0') is used as video_id
    video_dir = os.path.dirname(video_path)
    video_id = os.path.basename(video_dir)

    print(f'Processing {video_id} ...')
    detections = run_detection_on_video(video_path, model, conf=0.25)

    submission.append({
        'video_id': video_id,
        'detections': detections,  # [] if no detections, or list of {bboxes: [...]}
    })

# Save submission.json in the required format
with open(OUTPUT_PATH, 'w', encoding='utf-8') as f:
    json.dump(submission, f, indent=2)

print(f'Saved submission to: {OUTPUT_PATH}')
